*0.3 Classical NLP*

# POS tagging (spaCy)

**The situation.** A feature request: "show what customers are *asking us to do*". The verbs. Another: filter the word cloud to things (nouns) and drop the rest. And lemmatization (item 8) needed to know that "better" was an adjective. All three need the same thing: each word's part of speech.

**Part-of-speech tagging.** A model labels every token as NOUN, VERB, ADJ, ADV, PRON and so on. spaCy does it in the same pass as everything else — it is already computed when you call `nlp(text)`.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import spacy

nlp = spacy.load("en_core_web_sm")
ticket = "Please cancel my subscription and refund the last invoice quickly"
document = nlp(ticket)
print(f"{'token':<14}{'pos':>6}   {'fine tag':<8} meaning")
verbs = []
for token in document:
    if token.pos_ == "VERB":
        verbs.append(token.lemma_)
    print(f"{token.text:<14}{token.pos_:>6}   {token.tag_:<8} {spacy.explain(token.tag_)}")
print("what the customer asks us to do:", verbs)
assert verbs == ["cancel", "refund"]

token            pos   fine tag meaning
Please          INTJ   UH       interjection
cancel          VERB   VB       verb, base form
my              PRON   PRP$     pronoun, possessive
subscription    NOUN   NN       noun, singular or mass
and            CCONJ   CC       conjunction, coordinating
refund          VERB   VB       verb, base form
the              DET   DT       determiner
last             ADJ   JJ       adjective (English), other noun-modifier (Chinese)
invoice         NOUN   NN       noun, singular or mass
quickly          ADV   RB       adverb
what the customer asks us to do: ['cancel', 'refund']


**Reading the output.** Every word has a coarse tag (`VERB`, `NOUN`) and a fine one (`VB` base-form verb, `NNS` plural noun). The two verbs — cancel, refund — are the actions requested. "Please" is an interjection and "quickly" an adverb, so neither pollutes the list.

**Requested actions across many tickets.**

In [3]:
from collections import Counter

tickets = [
    "Please cancel my subscription.",
    "Can you refund the duplicate charge?",
    "I want to cancel and get a refund.",
    "Update my billing address.",
    "Cancel the renewal, it charged me twice.",
]
actions = Counter()
for document in nlp.pipe(tickets):
    for token in document:
        if token.pos_ == "VERB" and token.dep_ in (
            "ROOT",
            "conj",
            "xcomp",
            "advcl",
        ):  # the main verbs, not helpers
            actions[token.lemma_] += 1
print("requested actions:", actions.most_common())
assert actions["cancel"] >= 3

requested actions: [('cancel', 3), ('refund', 1), ('want', 1), ('get', 1), ('update', 1), ('charge', 1)]


**The rule to remember.** POS tags are the cheapest structure you can add to text: verbs for actions, nouns for things, adjectives for sentiment words. They come free with every spaCy call.

| Use it when | Don't when | Instead use |
|---|---|---|
| filtering words by role; features for classical models; lemmatization | you need the *meaning* of the request, not its grammar | an LLM classification or extraction |

**Watch out**
- Chat text with no capitals or punctuation drops tagger accuracy noticeably.
- "refund" is a noun in "get a refund" and a verb in "refund me" — the tagger decides by context, and short inputs give little.
- `token.dep_` (the dependency label) is the next step up: who does what to whom.